# 🐄 Notebook 2: Thundering Herd & Why Jitter Helps

Imagine 100 clients calling a service. The service blips for 1 second.
All 100 retry using *the same* exponential backoff — so they all retry **at the same moments**.
The service sees repeating spikes of 100 concurrent requests — a **stampede** (aka the "thundering herd").

With **jitter** (a little randomness), each client waits a slightly different amount, spreading the load over time.

In this notebook we simulate both scenarios and **see** the difference.

## 🛠️ Setup

```bash
cd 05-microservices/retry
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## Simulating 100 clients retrying together

Each client fails at t=0, then retries up to 5 times with exponential backoff starting at 500ms.
We count how many retries land in each 100ms time "bucket".
Many clients in the same bucket = a spike = the **herd**.

For jitter we use **full jitter**: sleep a random time in `[0, nominal_delay]`.
This is the variant recommended in AWS's *"Exponential Backoff And Jitter"* post.

In [ ]:
import random, collections

def simulate(jitter=False, clients=100, attempts=5, base=0.5, factor=2.0):
    """Return {bucket_time_sec: how_many_clients_retry_at_that_time}."""
    buckets = collections.Counter()
    for _ in range(clients):
        t = 0.0
        for a in range(1, attempts + 1):
            delay = base * (factor ** (a - 1))
            if jitter:
                # 'Full jitter': sleep uniformly in [0, delay].
                delay = random.uniform(0, delay)
            t += delay
            buckets[round(t, 1)] += 1  # group into 100ms buckets
    return buckets

def show(label, buckets, limit=20):
    print(f'--- {label} ---')
    peak = max(buckets.values()) if buckets else 0
    print(f'peak concurrent retries in one 100ms bucket: {peak}')
    for k, v in sorted(buckets.items())[:limit]:
        bar = '#' * v
        print(f'  t={k:>5.1f}s  {v:>3} {bar}')

random.seed(0)
show('NO jitter  (everyone retries at the same moments)', simulate(jitter=False))
print()
show('WITH jitter (full-jitter exponential backoff)', simulate(jitter=True), limit=40)


### What you should see

- **No jitter**: tall spikes of exactly 100 clients at `0.5s`, `1.5s`, `3.5s`, `7.5s`, `15.5s`.
  The downstream sees *the entire herd* land five times in a row — each spike is as bad as
  the original failure that triggered the retries.
- **With jitter**: peak concurrent retries drops dramatically, spread across a wide window.
  The service can actually recover while clients trickle in.

**The key number**: `peak concurrent retries` — that's the load the service has to survive.

## Why it's worse than it looks

The peak isn't just 100 — it's 100 retries *on top of* whatever fresh traffic is arriving.
If your normal load is 100 req/sec and everyone retries 5 times, your service may see
up to `100 new + 100 x 5 retries = 600 req/sec` right when it's already sick.

This is how a small blip turns into an outage — retries **amplify** failure.

Let's measure how much jitter reduces the peak at different scales:

In [ ]:
for n in (10, 100, 1000):
    random.seed(42)
    no_j = max(simulate(jitter=False, clients=n).values())
    random.seed(42)
    wj   = max(simulate(jitter=True,  clients=n).values())
    print(f'{n:>5} clients -> peak NO-jitter={no_j:>4}   '
          f'peak WITH-jitter={wj:>4}   '
          f'-> jitter reduces peak by {no_j/wj:.1f}x')


## Rules of thumb

- ✅ Always combine jitter with a **max delay cap** (e.g. 5–30s). Otherwise attempt 10 sleeps for ~17 minutes.
- ✅ Use a **max attempts** limit (e.g. 3–5) to avoid retry storms.
- ✅ Add a **per-process circuit breaker** so you stop piling on a dead dependency.
- ✅ Prefer **full jitter** (`random.uniform(0, delay)`) — simple and often best in practice.
  See AWS Architecture Blog: *"Exponential Backoff And Jitter"*.

Next: **Notebook 3** shows the rest of production retry — error classification, `Retry-After`, deadlines, retry budgets, idempotency keys, and per-attempt timeouts.